In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d
import pandas as pd
import xarray as xr

In [ ]:
df = pd.read_csv("../../data/LeafRiverDaily.csv")


L = df.shape[0]

print(L)
dx = xr.DataArray(df, dims=["Days", "Variables"], coords={"Days": range(L), "Variables" : ["Precipitation", "Temperature", "Streamflow"]})
dx

In [ ]:
def inerp_T_P(t_force, P, T, kind="linear", extrapolate=False):
    t_array = np.asarray(t_force)
    P_array = np.asarray(P)
    T_array = np.asarray(T)
    if t_force.ndim != 1:
        raise ValueError("t_force must be 1D array")
    if P.shape != t_force.shape or T.shape != t_force.shape:
        raise ValueError("t_force, T, and P must be same shape")
    sort = np.argsort(t_force)
    t_sort = t_array[sort]
    P_sort = P_array[sort]
    T_sort = T_array[sort]
    if extrapolate:
        fillP = 'extrapolate'
        fillT = 'extrapolate'
    else:
        fillP = (P_sort[0], P_sort[-1])
        fillT = (T_sort[0], T_sort[-1])


    P_t = interp1d(t_sort, P_sort, kind = kind, fill_value = fillP, assume_sorted=True)
    T_t = interp1d(t_sort, T_sort, kind = kind, fill_value = fillT, assume_sorted=True)

    return P_t, T_t


In [ ]:
def make_ode(P_t, T_t, a, b, c, s_cutoff=True):
    a = float(a)
    b = float(b)
    c = float(c)
    def ode(t, y):
        S = float(y[0])

        P = float(P_t[t])
        T = float(T_t[t])

        T_pos = max(T, 0.0)

        if s_cutoff:
            S_eff = max(S, 0.0)
        else:
            S_eff = S

        dSdt = P-a* T_pos - b * (S_eff**c)
        return [dSdt]
    return ode

In [ ]:
t_f = np.linspace(0,dx.shape[0], dx.shape[0], dtype = int)

p_force = dx.sel(Variables="Precipitation").to_numpy()
temp_force = dx.sel(Variables="Temperature").to_numpy()


p_t, t_t = inerp_T_P(t_f, p_force, temp_force)


a = 0.8
b = 0.003
c = 2.7

ode = make_ode(p_force, temp_force, a, b, c)


In [ ]:
S0 = [0.0]

sol = solve_ivp(ode, t_span=(t_f[0], t_f[-1]), y0=S0, t_eval=t_f, method="RK45", rtol=1e-6, atol=1e-9)